# 新規100候補：抽出・検証・品質評価

**90候補を実装、10候補は役・点数・完全振聴判定が未対応です。全100候補の正確性が確認済みという意味ではありません。**
実牌譜の検証はこのNotebookで行います。モデル学習はありません。既存20＋3特徴量、C1〜C10、H1〜H6は変更しません。

最初はN1→N2（A）→N3→N4→N5。監査を確認後、N2でBへ変更してN3〜N5。全件完了後にN6のCを実行します。
段位コード16以下／18以上、17除外、初期1500対局と同じ時間帯の除外、共通対局除外を維持します。


## N1：Drive接続・固定版取得・合成テスト

依存ライブラリは別フォルダ・別プロセスで使用します。実牌譜はここでは読みません。

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, os, subprocess, json, urllib.request, hashlib
CODE_COMMIT = 'd6b112589536cdf94f78e59993c39fb22da1b1d8'
SOURCE = Path('/tmp') / ('features_100_' + CODE_COMMIT[:12])
FILES = ['choices/choices.py', 'decision_features/__init__.py', 'decision_features/definitions.py', 'decision_features/extract.py', 'decision_features/pipeline.py', 'feature_design_100/feature_candidates_100.csv', 'features_100/__init__.py', 'features_100/calls.py', 'features_100/cli.py', 'features_100/extract.py', 'features_100/options.py', 'features_100/pipeline.py', 'features_100/quality.py', 'features_100/requirements.txt', 'features_100/rules.py', 'features_100/shapes.py', 'features_100/tests/__init__.py', 'features_100/tests/test_core.py', 'features_100/tests/test_integration.py', 'features_100/tests/test_quality.py', 'features_100/tests/test_rule_tables.py', 'mahjong_research/__init__.py', 'mahjong_research/analysis.py', 'mahjong_research/inputs.py', 'mahjong_research/legacy.py', 'mahjong_research/tenhou.py', 'smoothing/study.py']
for relative in FILES:
    target = SOURCE / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{CODE_COMMIT}/research_v2/{relative}'
    content = urllib.request.urlopen(url).read()
    if target.exists() and target.read_bytes() != content: raise RuntimeError('固定版と不一致: ' + relative)
    if not target.exists(): target.write_bytes(content)
DEPS = SOURCE / '_dependencies'
stamp = DEPS / 'install_complete.txt'
requirements = SOURCE / 'features_100/requirements.txt'
fingerprint = hashlib.sha256(requirements.read_bytes()).hexdigest()
if not stamp.exists():
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(DEPS),'-r',str(requirements)])
    stamp.write_text(fingerprint)
elif stamp.read_text() != fingerprint: raise RuntimeError('依存版が不一致。新しいセッションを使用してください。')
ENV = dict(os.environ, PYTHONPATH=os.pathsep.join(map(str,[DEPS,SOURCE,SOURCE/'choices',SOURCE/'smoothing'])), PYTHONIOENCODING='utf-8', MPLBACKEND='Agg')
subprocess.check_call([sys.executable,'-m','unittest','discover','-s',str(SOURCE/'features_100/tests'),'-t',str(SOURCE),'-q'],env=ENV)
# 日本語図用フォント。解析結果には影響しません。
subprocess.run(['apt-get','install','-y','-qq','fonts-noto-cjk'],check=False,stdout=subprocess.DEVNULL)
print('固定版取得・合成テスト完了。実牌譜の検証はN4以降です。')


Mounted at /content/drive
固定版取得・合成テスト完了。実牌譜の検証はN4以降です。


## N2：入力・モード・出力の設定

RUN_DIRはinput_plan.jsonとshared_games.csvがある既存実行フォルダ。C5キャッシュの場所は計画ファイルから読みます。
元ZIPが移動していればZIP_OVERRIDEを変更してください。PLAYERは完全一致名です（全角ｎ）。
Aは各群1〜10対局、または明示ID最大60件。Bは全C5対局を抽出し、17・初期1500等の除外は窓作成時に適用します。
新出力の日時フォルダは毎回作成します。同じ版・元XML・対象の成功キャッシュだけ再利用します。


In [2]:
ROOT = Path('/content/drive/MyDrive/卒論')
RUN_DIR = ROOT / 'research_v2_runs/20261105T071722Z_5da6c9ad'
OUTPUT_ROOT = ROOT / 'research_v2_features_100'
PLAYER = 'しょぼんｎ'
KEY = 'shobon'
ZIP_OVERRIDE = None
MODE = 'A'  # 最初はA。監査確認後にBへ変更。
PILOT_PER_GROUP = 3
PILOT_IDS = None  # 指定するならC5のgame_idを重複なく列挙
EXPLORATION_END_HOUR = ''  # C実行前に例 '2014-01-01 00:00:00' のように正時の境界を指定。例は研究上の推薦ではない。
CONFIG = SOURCE / 'config.json'
RECEIPT = SOURCE / ('preview_' + MODE + '.json')
def save_config():
    CONFIG.write_text(json.dumps(dict(run_dir=str(RUN_DIR),output_root=str(OUTPUT_ROOT),player=PLAYER,key=KEY,
        zip_override=ZIP_OVERRIDE,pilot_per_group=PILOT_PER_GROUP,pilot_ids=PILOT_IDS,
        exploration_end_hour=EXPLORATION_END_HOUR),ensure_ascii=False),encoding='utf8')
def run_step(action,extra=()):
    subprocess.check_call([sys.executable,'-m','features_100.cli',action,str(CONFIG),'--mode',MODE,
        '--receipt',str(RECEIPT),*extra],env=ENV)
save_config()
print('入力:',RUN_DIR,'\n出力親:',OUTPUT_ROOT,'\nモード:',MODE)


入力: /content/drive/MyDrive/卒論/research_v2_runs/20261105T071722Z_5da6c9ad 
出力親: /content/drive/MyDrive/卒論/research_v2_features_100 
モード: A


## N3：入力監査と処理件数の表示（抽出なし）

C5署名・ハッシュ・ID集合・ZIP内の対象名を確認します。表示件数を確認してN4へ進みます。

In [3]:
run_step('preview')


## N4：Aの小規模検証／Bの全件抽出

全実装済み候補を共通の状態復元から計算します。初回は時間がかかるためAの実測秒数を確認してください。
中断後は同じ固定版と設定でN1〜N4を再実行。成功キャッシュは再利用し、失敗対局は監査に残します。
Aは判断直前の状態・X/Y・実行・理由をtracesに保存します。BキャッシュからAの詳細監査を追加する場合だけ指定対局を読み直します。


In [4]:
run_step('extract')
EXTRACTION_DIR = Path(json.loads(RECEIPT.with_suffix('.result.json').read_text())['output'])
print('抽出結果:', EXTRACTION_DIR)


抽出結果: /content/drive/MyDrive/卒論/research_v2_features_100/20261126T044531Z_84d224bf


## N5：監査・分子分母の確認

保留10候補はunimplemented、機会ゼロはno_opportunity、不明条件を含む場合はunknown。
unknownの観測済みカウンタは部分集計であり、完成した割合として使用しません。
各候補の正例が出ない場合は未検証です。Aの少数対局を詰めて10/20対局窓を作りません。


In [6]:
import pandas as pd
from IPython.display import display, Markdown
for name in ['implementation_status.csv','extraction_audit.csv','feature_audit.csv']:
    print(name)
    display(pd.read_csv(EXTRACTION_DIR/name))
display(Markdown((EXTRACTION_DIR/'report_ja.md').read_text(encoding='utf8')))
print('Aの判断記録:', EXTRACTION_DIR/'traces')


implementation_status.csv


,feature_id,english_name,japanese_name,domain,implementation,reason,real_log_validated,definition_commit
0,P01,tenpai_unsafe_preserving_choice,現物保有時の非現物テンパイ選択,押し引き判断,implemented_pilot_required,合成検証のみ。実牌譜での正例・分母・選択肢の照合は未実施,False,2079e5ba5af5e3401ba458dd27d464a5f717b5f9
1,P02,one_shanten_unsafe_preserving_choice,現物保有時の非現物1シャンテン維持選択,押し引き判断,implemented_pilot_required,合成検証のみ。実牌譜での正例・分母・選択肢の照合は未実施,False,2079e5ba5af5e3401ba458dd27d464a5f717b5f9
2,P03,far_hand_unsafe_preserving_choice,現物保有時の非現物遠手維持選択,押し引き判断,implemented_pilot_required,合成検証のみ。実牌譜での正例・分母・選択肢の照合は未実施,False,2079e5ba5af5e3401ba458dd27d464a5f717b5f9
3,P04,tenpai_safety_conflict_preserve_choice,現物と競合するテンパイ形の選択,押し引き判断,implemented_pilot_required,合成検証のみ。実牌譜での正例・分母・選択肢の照合は未実施,False,2079e5ba5af5e3401ba458dd27d464a5f717b5f9
4,P05,one_shanten_safety_conflict_preserve_choice,現物と競合する1シャンテン維持選択,押し引き判断,implemented_pilot_required,合成検証のみ。実牌譜での正例・分母・選択肢の照合は未実施,False,2079e5ba5af5e3401ba458dd27d464a5f717b5f9
...,...,...,...,...,...,...,...,...
95,S06,dealer_repeat_yakuhai_call_choice,連荘中親の役牌進行ポン選択,局面・点数状況,implemented_pilot_required,合成検証のみ。実牌譜での正例・分母・選択肢の照合は未実施,False,2079e5ba5af5e3401ba458dd27d464a5f717b5f9
96,S07,riichi_stick_pool_call_choice,供託が多い局の進行改善鳴き選択,局面・点数状況,implemented_pilot_required,合成検証のみ。実牌譜での正例・分母・選択肢の照合は未実施,False,2079e5ba5af5e3401ba458dd27d464a5f717b5f9
97,S08,target_opponent_dama_wait_choice,最終予定局で逆転可能相手への役あり待ち選択,局面・点数状況,deferred,fixed_yaku_score_or_complete_furiten_not_imple...,False,2079e5ba5af5e3401ba458dd27d464a5f717b5f9
98,S09,score_gap_dora_ukeire_choice,最終予定局で表ドラ保持と受入が競合,局面・点数状況,implemented_pilot_required,合成検証のみ。実牌譜での正例・分母・選択肢の照合は未実施,False,2079e5ba5af5e3401ba458dd27d464a5f717b5f9


extraction_audit.csv


,game_id,file_id,xml_sha256,seat,dan,player,hour,status,error,reused,trace_only,seconds,rule_profile_verified,rounds,own_discards,calls,kans
0,2012062304gm-0029-0000-dff1fa7e,mjlog_pf4-20_n30/2012062304gm-0029-0000-dff1fa...,cb37e2e6bd8949bb661decb8e879563fe5362e52995f70...,0,14,しょぼんｎ,2012-06-23 04:00:00,ok,NaN,False,False,20.902308,True,11,130,20,NaN
1,2014061300gm-00a9-0000-95909518,mjlog_pf4-20_n30/2014061300gm-00a9-0000-959095...,86a36c8b1c97a3f16d8b5f739d0b0d4858fdc121d3e7aa...,0,18,しょぼんｎ,2014-06-13 00:00:00,ok,NaN,False,False,13.990215,True,10,122,25,2.0
2,2021050905gm-00a9-0000-6aeb09b8,mjlog_pf4-20_n30/2021050905gm-00a9-0000-6aeb09...,f1b3429b79ca63a10c1a3b31e2eb6f2ed53b669f80257b...,2,16,しょぼんｎ,2021-05-09 05:00:00,ok,NaN,False,False,33.969928,True,14,196,29,1.0
3,2024062620gm-00a9-0000-4eb98c09,mjlog_pf4-20_n30/2024062620gm-00a9-0000-4eb98c...,c70048ab69bf65d28c5be1ffb55b6db9389554a26dbfa9...,1,18,しょぼんｎ,2024-06-26 20:00:00,ok,NaN,False,False,12.608408,True,11,141,16,3.0
4,2025100518gm-0029-0000-8c403a29,mjlog_pf4-20_n30/2025100518gm-0029-0000-8c403a...,5834e6af7c25e2b621d71188e9b628defa13e24264b53a...,1,16,しょぼんｎ,2025-10-05 18:00:00,ok,NaN,False,False,6.668725,True,9,98,22,NaN
5,2026051522gm-00a9-0000-0925ded1,mjlog_pf4-20_n30/2026051522gm-00a9-0000-0925de...,47e14f505aeaa3190dfedae7d7636ff8edc33d33032863...,1,20,しょぼんｎ,2026-05-15 22:00:00,ok,NaN,False,False,12.320452,True,9,100,20,1.0


feature_audit.csv


,feature_id,games,reconstruction_ok,extraction_errors,unknown_games,unimplemented_games,zero_denominator_games,observed_numerator,observed_denominator,observed_totals_are_partial,censored_responses
0,P01,6,6,0,0,0,3,13.0,19.0,False,0.0
1,P02,6,6,0,0,0,1,9.0,22.0,False,0.0
2,P03,6,6,0,0,0,0,16.0,56.0,False,0.0
3,P04,6,6,0,0,0,3,13.0,17.0,False,0.0
4,P05,6,6,0,0,0,3,7.0,10.0,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...
102,BASE_FM_T_UNSAFE,6,6,0,0,0,5,0.0,1.0,False,0.0
103,BASE_F0_MAX_U,6,6,0,0,0,0,246.0,409.0,False,0.0
104,BASE_F1_SAFE,6,6,0,0,0,0,59.0,102.0,False,0.0
105,BASE_F0_DORA_VS_U,6,6,0,0,0,3,6.0,9.0,False,0.0


# 抽出監査

対象6対局、復元成功6。実装90、保留10候補。

分母ゼロは欠損、unknownを含む割合も欠損です。実装済みは実牌譜検証済みを意味しません。feature_audit.csv、extraction_audit.csvとAのtracesを確認してください。局内反復は独立試行ではありません。同一時間帯内の順序・全履歴完全性は未確認です。Cは探索期間を指定し別フォルダへ保存します。


Aの判断記録: /content/drive/MyDrive/卒論/research_v2_features_100/20261126T044531Z_84d224bf/traces


In [7]:
# N7：抽出処理の詳細プロファイリング
# 既存の研究コード・Driveの保存結果は変更しない

import sys
import os
import json
import time
import tempfile
import subprocess
from pathlib import Path

assert MODE == "A", "N2のMODEをAにしてください"
assert RECEIPT.exists(), "先にN3を実行してください"
assert EXTRACTION_DIR.exists(), "先にN4を実行してください"

PROFILE_DIR = Path(tempfile.mkdtemp(prefix="mahjong_profile_"))
PROFILE_SCRIPT = PROFILE_DIR / "profile_extract.py"

PROFILE_SCRIPT.write_text(r'''
import sys
import json
import time
import zipfile
import cProfile
import pstats
from pathlib import Path

from features_100.pipeline import prepare, checked_xml
from features_100.extract import extract
from mahjong_research.inputs import digest

receipt = Path(sys.argv[1])
original_dir = Path(sys.argv[2])
work_dir = Path(sys.argv[3])
mode = sys.argv[4]

original = json.loads(receipt.read_text(encoding="utf8"))
config = original["config"]

# 元の6対局と同一の入力・設定であることを確認
started = time.perf_counter()
ctx = prepare(config, "A")
prepare_seconds = time.perf_counter() - started

expected = dict(
    signature=ctx["signature"],
    mode="A",
    config=config,
    ids=ctx["targets"].game_id.tolist()
)

if original != expected:
    raise RuntimeError("元のプレビューと入力・設定が異なります")

# 低段位期・高段位期から1対局ずつ選択
rows = list(ctx["targets"].itertuples())
low = [r for r in rows if int(r.dan) <= 16]
high = [r for r in rows if int(r.dan) >= 18]

assert low and high, "両段位期の対局が必要です"
selected = [low[0], high[0]]

if mode == "PROFILE":
    b_times = json.loads(
        (work_dir / "timing_B.json").read_text()
    )
    slowest = max(b_times, key=lambda x: x["seconds"])
    selected = [
        r for r in selected
        if str(r.game_id) == slowest["game_id"]
    ]

print("\n実行モード:", mode, flush=True)
print("入力準備時間:", round(prepare_seconds, 3), "秒", flush=True)

timings = []

with zipfile.ZipFile(ctx["archive"]) as z:
    for r in selected:
        # 既存のキャッシュを使用せず、元XMLを読む
        t = time.perf_counter()
        xml = checked_xml(z, r)
        read_seconds = time.perf_counter() - t

        print(
            "\n対局:", r.game_id,
            "段位コード:", r.dan,
            flush=True
        )

        profiler = cProfile.Profile()

        if mode == "PROFILE":
            profiler.enable()

        t = time.perf_counter()

        result = extract(
            xml,
            config["player"],
            r.seat,
            r.dan,
            r.game_id,
            collect_trace=(mode == "A")
        )

        elapsed = time.perf_counter() - t

        if mode == "PROFILE":
            profiler.disable()

        # N4で保存した元の計算結果と比較
        old_path = (
            original_dir / "traces" /
            (digest(r.game_id) + ".json")
        )

        old_packet = json.loads(
            old_path.read_text(encoding="utf8")
        )

        assert old_packet["identity"]["game_id"] == r.game_id
        assert old_packet["identity"]["xml_sha256"] == r.xml_sha256

        old = old_packet["result"]["features"]
        new = result["features"]

        differences = [
            key for key in sorted(set(old) | set(new))
            if old.get(key) != new.get(key)
        ]

        print("XML読込・照合:", round(read_seconds, 3), "秒")
        print("特徴量抽出:", round(elapsed, 3), "秒")
        print("比較対象項目数:", len(old))
        print("不一致:", len(differences))
        print("局数:", result["audit"]["rounds"])

        if differences:
            for key in differences[:10]:
                print(key, "旧:", old.get(key), "新:", new.get(key))
            raise RuntimeError("保存結果との不一致を検出しました")

        timings.append(dict(
            game_id=str(r.game_id),
            dan=int(r.dan),
            seconds=elapsed,
            xml_seconds=read_seconds
        ))

        if mode == "PROFILE":
            print("\n=== 累積処理時間 上位25件 ===")
            pstats.Stats(profiler).strip_dirs().sort_stats(
                "cumulative"
            ).print_stats(25)

            print("\n=== 関数自身の処理時間 上位20件 ===")
            pstats.Stats(profiler).strip_dirs().sort_stats(
                "tottime"
            ).print_stats(20)

    # 既存の計算キャッシュの利用状況
    from features_100 import shapes

    print("\n=== キャッシュ統計 ===")
    for name in ["partitions", "shanten", "ukeire", "improvement"]:
        fn = getattr(shapes, name)
        print(name, fn.cache_info())

(work_dir / f"timing_{mode}.json").write_text(
    json.dumps(timings, ensure_ascii=False),
    encoding="utf8"
)
''', encoding="utf8")

print("CPU数:", os.cpu_count())
print("固定版:", CODE_COMMIT)
print("既存抽出結果:", EXTRACTION_DIR)

# 各モードを独立したPythonプロセスで実行
for profile_mode in ["A", "B", "PROFILE"]:
    print("\n" + "=" * 60, flush=True)
    print("開始:", profile_mode, flush=True)

    subprocess.check_call(
        [
            sys.executable,
            "-u",
            str(PROFILE_SCRIPT),
            str(RECEIPT),
            str(EXTRACTION_DIR),
            str(PROFILE_DIR),
            profile_mode
        ],
        env=ENV
    )

# AとBの実測比較
a = json.loads((PROFILE_DIR / "timing_A.json").read_text())
b = json.loads((PROFILE_DIR / "timing_B.json").read_text())

import pandas as pd

comparison = pd.DataFrame([
    {
        "game_id": x["game_id"],
        "段位コード": x["dan"],
        "A_秒": round(x["seconds"], 3),
        "B_秒": round(y["seconds"], 3),
        "Bの高速化倍率": round(x["seconds"] / y["seconds"], 3)
    }
    for x, y in zip(a, b)
])

print("\n=== 最終比較結果 ===")
display(comparison)

print(
    "A平均:", round(comparison["A_秒"].mean(), 3), "秒"
)
print(
    "B平均:", round(comparison["B_秒"].mean(), 3), "秒"
)
print("\n計測完了")


CPU数: 2
固定版: d6b112589536cdf94f78e59993c39fb22da1b1d8
既存抽出結果: /content/drive/MyDrive/卒論/research_v2_features_100/20261126T044531Z_84d224bf

開始: A

開始: B

開始: PROFILE

=== 最終比較結果 ===


,game_id,段位コード,A_秒,B_秒,Bの高速化倍率
0,2012062304gm-0029-0000-dff1fa7e,14,19.105,18.144,1.053
1,2014061300gm-00a9-0000-95909518,18,11.757,12.041,0.976


A平均: 15.431 秒
B平均: 15.092 秒

計測完了


In [8]:
# N8：詳細プロファイリング結果の取得

result = subprocess.run(
    [
        sys.executable,
        "-u",
        str(PROFILE_SCRIPT),
        str(RECEIPT),
        str(EXTRACTION_DIR),
        str(PROFILE_DIR),
        "PROFILE"
    ],
    env=ENV,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

(PROFILE_DIR / "profile_report.txt").write_text(
    result.stdout,
    encoding="utf-8"
)

print("\n終了コード:", result.returncode)

result.check_returncode()


A: 対象6 / キャッシュ候補6 / 新規解析予定0対局
実装90 / 保留10。保留項目を0で代用しません。
ZIP: /content/drive/MyDrive/卒論/mjlog_pf4-20_n30.zip
出力親: /content/drive/MyDrive/卒論/research_v2_features_100

実行モード: PROFILE
入力準備時間: 0.84 秒

対局: 2012062304gm-0029-0000-dff1fa7e 段位コード: 14
XML読込・照合: 0.001 秒
特徴量抽出: 44.875 秒
比較対象項目数: 107
不一致: 0
局数: 11

=== 累積処理時間 上位25件 ===
         41701751 function calls (39933202 primitive calls) in 43.184 seconds

   Ordered by: cumulative time
   List reduced from 228 to 25 due to restriction <25>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.058    0.058   44.874   44.874 extract.py:24(extract)
2487/1545    0.008    0.000   44.107    0.029 functools.py:1009(__get__)
    13910    0.071    0.000   43.788    0.003 rules.py:188(measure)
    10000    0.074    0.000   43.715    0.004 rules.py:20(evaluate)
     3173    0.004    0.000   39.820    0.013 options.py:118(max)
     3173    0.002    0.000   39.800    0.013 {built-in method builtins.getattr}
    11768  

In [9]:
# N9：牌種別分解による高速化実験
# GitHub・既存のDrive保存結果は変更しない

FAST_SCRIPT = PROFILE_DIR / "fast_experiment.py"

FAST_SCRIPT.write_text(r'''
import sys
import json
import time
import zipfile
import random
from pathlib import Path
from functools import lru_cache

from features_100 import shapes
from features_100.pipeline import prepare, checked_xml
from features_100.extract import extract
from mahjong_research.inputs import digest

legacy_partitions = shapes.partitions


def prune(out):
    best = {}
    for m, b, p in out:
        key = (m, p)
        best[key] = max(b, best.get(key, -1))
    return tuple(
        (m, b, p)
        for (m, p), b in sorted(best.items())
    )


@lru_cache(maxsize=100000)
def block(c, suited):
    """1牌種ブロック内で旧版と同じ分解を列挙する。"""
    i = next((i for i, n in enumerate(c) if n), None)

    if i is None:
        return ((0, 0, 0),)

    # 孤立牌として使用
    a = list(c)
    a[i] -= 1
    out = set(block(tuple(a), suited))

    # 刻子
    if c[i] >= 3:
        a = list(c)
        a[i] -= 3
        for m, b, p in block(tuple(a), suited):
            if m < 4:
                out.add((m + 1, b, p))

    # 対子：雀頭または搭子
    if c[i] >= 2:
        a = list(c)
        a[i] -= 2
        for m, b, p in block(tuple(a), suited):
            out.add((m, b + 1, p))
            if p == 0:
                out.add((m, b, 1))

    if suited:
        # 順子
        if i + 2 < len(c) and c[i+1] and c[i+2]:
            a = list(c)
            for t in (i, i+1, i+2):
                a[t] -= 1
            for m, b, p in block(tuple(a), suited):
                if m < 4:
                    out.add((m + 1, b, p))

        # 連続・嵌張搭子
        for delta in (1, 2):
            j = i + delta
            if j < len(c) and c[j]:
                a = list(c)
                a[i] -= 1
                a[j] -= 1
                for m, b, p in block(tuple(a), suited):
                    out.add((m, b + 1, p))

    return prune(out)


@lru_cache(maxsize=100000)
def merge(left, right):
    """異なる牌種の分解結果を結合。"""
    best = {}

    for m1, b1, p1 in left:
        for m2, b2, p2 in right:
            m = m1 + m2
            p = p1 + p2

            if m > 4 or p > 1:
                continue

            key = (m, p)
            b = b1 + b2

            if b > best.get(key, -1):
                best[key] = b

    return tuple(
        (m, b, p)
        for (m, p), b in sorted(best.items())
    )


@lru_cache(maxsize=120000)
def fast_partitions(c):
    man = block(c[:9], True)
    pin = block(c[9:18], True)
    sou = block(c[18:27], True)
    honors = block(c[27:], False)

    return merge(
        merge(man, pin),
        merge(sou, honors)
    )


# ========================================
# 1. 合成手牌による旧新版の一致確認
# ========================================

print("=== 合成手牌テスト ===", flush=True)

rng = random.Random(20261126)

hands = [
    (0,) * 34,
    (1,) * 14 + (0,) * 20,
    (2,) * 7 + (0,) * 27,
    (4, 4, 4, 2) + (0,) * 30,
    (0,) * 27 + (2,) * 7,
    (0,) * 27 + (4, 4, 4, 2, 0, 0, 0),
]

for _ in range(200):
    n = rng.randint(1, 14)
    tiles = rng.sample(range(136), n)
    c = [0] * 34
    for tile in tiles:
        c[tile // 4] += 1
    hands.append(tuple(c))

for index, c in enumerate(hands):
    old = legacy_partitions(c)
    new = fast_partitions(c)

    if old != new:
        print("不一致手牌:", c)
        print("旧版:", old)
        print("新版:", new)
        raise RuntimeError("分解結果不一致")

print(
    f"合成手牌: {len(hands)}件すべて一致",
    flush=True
)

# 計測前にテストで使用したキャッシュを消す
for fn in [
    legacy_partitions,
    fast_partitions,
    block,
    merge,
    shapes.shanten,
    shapes.ukeire,
    shapes.improvement
]:
    fn.cache_clear()

# 分解関数だけを、このプロセス内で差し替える
shapes.partitions = fast_partitions


# ========================================
# 2. 元の6対局を再解析
# ========================================

receipt = Path(sys.argv[1])
original_dir = Path(sys.argv[2])
profile_dir = Path(sys.argv[3])

original = json.loads(receipt.read_text(encoding="utf8"))
config = original["config"]

ctx = prepare(config, "A")

expected = dict(
    signature=ctx["signature"],
    mode="A",
    config=config,
    ids=ctx["targets"].game_id.tolist()
)

if original != expected:
    raise RuntimeError("元の入力・設定と不一致")

rows = list(ctx["targets"].itertuples())

assert len(rows) == 6

# N7で測った2試合を先に実行する
baseline = json.loads(
    (profile_dir / "timing_B.json").read_text()
)

baseline_times = {
    x["game_id"]: x["seconds"]
    for x in baseline
}

baseline_order = {
    x["game_id"]: i
    for i, x in enumerate(baseline)
}

rows.sort(
    key=lambda r: baseline_order.get(str(r.game_id), 999)
)

print("\n=== 実牌譜6対局の比較 ===", flush=True)

results = []

with zipfile.ZipFile(ctx["archive"]) as z:
    for index, r in enumerate(rows, 1):

        xml = checked_xml(z, r)

        t = time.perf_counter()

        result = extract(
            xml,
            config["player"],
            r.seat,
            r.dan,
            r.game_id,
            collect_trace=False
        )

        elapsed = time.perf_counter() - t

        old_path = (
            original_dir / "traces" /
            (digest(r.game_id) + ".json")
        )

        old_packet = json.loads(
            old_path.read_text(encoding="utf8")
        )

        assert old_packet["identity"]["game_id"] == r.game_id
        assert old_packet["identity"]["xml_sha256"] == r.xml_sha256

        old = old_packet["result"]["features"]
        new = result["features"]

        differences = [
            k for k in sorted(set(old) | set(new))
            if old.get(k) != new.get(k)
        ]

        print(
            f"{index}/6 {r.game_id} "
            f"新版={elapsed:.3f}秒 "
            f"不一致={len(differences)}",
            flush=True
        )

        if differences:
            for k in differences[:10]:
                print(k, "旧版:", old.get(k), "新版:", new.get(k))
            raise RuntimeError("特徴量不一致")

        baseline_seconds = baseline_times.get(str(r.game_id))

        results.append({
            "game_id": str(r.game_id),
            "dan": int(r.dan),
            "new_seconds": elapsed,
            "old_seconds": baseline_seconds,
            "speedup": (
                baseline_seconds / elapsed
                if baseline_seconds is not None else None
            ),
            "compared_features": len(old),
            "differences": len(differences)
        })

print("\n=== キャッシュ統計 ===", flush=True)

for name, fn in [
    ("block", block),
    ("merge", merge),
    ("partitions", fast_partitions),
    ("shanten", shapes.shanten),
    ("ukeire", shapes.ukeire),
    ("improvement", shapes.improvement)
]:
    print(name, fn.cache_info(), flush=True)

(profile_dir / "fast_results.json").write_text(
    json.dumps(results, ensure_ascii=False),
    encoding="utf8"
)

print("\n6対局の一致確認完了", flush=True)
''', encoding="utf8")


# 別プロセスで実行し、出力を逐次表示
process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        str(FAST_SCRIPT),
        str(RECEIPT),
        str(EXTRACTION_DIR),
        str(PROFILE_DIR)
    ],
    env=ENV,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace"
)

for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()

if returncode != 0:
    raise RuntimeError(f"高速化実験失敗: {returncode}")


# ========================================
# 3. 速度比較結果
# ========================================

import pandas as pd

df = pd.DataFrame(json.loads(
    (PROFILE_DIR / "fast_results.json").read_text()
))

display(df.round(3))

paired = df.dropna(subset=["old_seconds"])

print("\n=== 同じ2対局の速度比較 ===")
print("旧版合計:", round(paired.old_seconds.sum(), 3), "秒")
print("新版合計:", round(paired.new_seconds.sum(), 3), "秒")

print(
    "高速化倍率:",
    round(
        paired.old_seconds.sum() /
        paired.new_seconds.sum(), 3
    ),
    "倍"
)

print(
    "新版6対局平均:",
    round(df.new_seconds.mean(), 3),
    "秒"
)

print("特徴量不一致合計:", df.differences.sum())


=== 合成手牌テスト ===
合成手牌: 206件すべて一致
A: 対象6 / キャッシュ候補6 / 新規解析予定0対局
実装90 / 保留10。保留項目を0で代用しません。
ZIP: /content/drive/MyDrive/卒論/mjlog_pf4-20_n30.zip
出力親: /content/drive/MyDrive/卒論/research_v2_features_100

=== 実牌譜6対局の比較 ===
1/6 2012062304gm-0029-0000-dff1fa7e 新版=6.536秒 不一致=0
2/6 2014061300gm-00a9-0000-95909518 新版=2.662秒 不一致=0
3/6 2021050905gm-00a9-0000-6aeb09b8 新版=7.638秒 不一致=0
4/6 2024062620gm-00a9-0000-4eb98c09 新版=4.475秒 不一致=0
5/6 2025100518gm-0029-0000-8c403a29 新版=1.732秒 不一致=0
6/6 2026051522gm-00a9-0000-0925ded1 新版=3.230秒 不一致=0

=== キャッシュ統計 ===
block CacheInfo(hits=3803539, misses=14497, maxsize=100000, currsize=14497)
merge CacheInfo(hits=2829673, misses=2669, maxsize=100000, currsize=2669)
partitions CacheInfo(hits=0, misses=944114, maxsize=120000, currsize=120000)
shanten CacheInfo(hits=1164215, misses=944114, maxsize=60000, currsize=60000)
ukeire CacheInfo(hits=16466, misses=57249, maxsize=40000, currsize=40000)
improvement CacheInfo(hits=0, misses=362, maxsize=6000, currsize=362)

6対局の一

,game_id,dan,new_seconds,old_seconds,speedup,compared_features,differences
0,2012062304gm-0029-0000-dff1fa7e,14,6.536,18.144,2.776,107,0
1,2014061300gm-00a9-0000-95909518,18,2.662,12.041,4.523,107,0
2,2021050905gm-00a9-0000-6aeb09b8,16,7.638,NaN,NaN,107,0
3,2024062620gm-00a9-0000-4eb98c09,18,4.475,NaN,NaN,107,0
4,2025100518gm-0029-0000-8c403a29,16,1.732,NaN,NaN,107,0
5,2026051522gm-00a9-0000-0925ded1,20,3.230,NaN,NaN,107,0



=== 同じ2対局の速度比較 ===
旧版合計: 30.185 秒
新版合計: 9.198 秒
高速化倍率: 3.282 倍
新版6対局平均: 4.379 秒
特徴量不一致合計: 0


In [10]:
# N10：シャンテン数・受け入れ枚数の追加高速化

FAST2_SCRIPT = PROFILE_DIR / "fast2_experiment.py"

# N9の牌種別分解・結合関数を再利用
prefix = FAST_SCRIPT.read_text(
    encoding="utf8"
).split("# 1. 合成手牌による旧新版の一致確認", 1)[0]

FAST2_SCRIPT.write_text(prefix + r'''

from features_100 import options, calls

shapes.partitions = fast_partitions

old_shanten = shapes.shanten
old_ukeire = shapes.ukeire


# 1. 面子手シャンテン数の共通計算

@lru_cache(maxsize=50000)
def standard_score(parts, fixed):
    return min(
        8 - 2*(m+fixed) - min(b, 4-m-fixed) - p
        for m, b, p in parts
        if m+fixed <= 4
    )


@lru_cache(maxsize=60000)
def faster_shanten(c, fixed=0):
    if (
        len(c) != 34
        or min(c) < 0
        or max(c) > 4
        or not 0 <= fixed <= 4
    ):
        raise ValueError("invalid shape counts")

    parts = merge(
        merge(block(c[:9], True), block(c[9:18], True)),
        merge(block(c[18:27], True), block(c[27:], False))
    )

    standard = standard_score(parts, fixed)

    if fixed:
        return standard, 99, 99

    distinct = 34 - c.count(0)
    pair_count = distinct - c.count(1)

    pairs = 6 - pair_count + max(0, 7-distinct)

    orphans = (
        13
        - sum(c[t] > 0 for t in shapes.TERMINALS)
        - int(any(c[t] >= 2 for t in shapes.TERMINALS))
    )

    return standard, pairs, orphans


# 2. 構造的受け入れと残り枚数の計算を分離

@lru_cache(maxsize=60000)
def acceptance_types(c, fixed):
    s = min(faster_shanten(c, fixed))
    a = list(c)
    accepted = []

    for t in range(34):
        if c[t] >= 4:
            continue

        a[t] += 1
        if min(faster_shanten(tuple(a), fixed)) < s:
            accepted.append(t)
        a[t] -= 1

    return tuple(accepted)


@lru_cache(maxsize=40000)
def faster_ukeire(c, fixed, known, owned_meld):
    return sum(
        4-known[t]
        for t in acceptance_types(c, fixed)
        if known[t] < 4
        and c[t]+owned_meld[t] < 4
    )


# 3. 合成手牌で旧新版を検証

rng = random.Random(20261126)

print("=== 合成手牌テスト ===", flush=True)

for i in range(300):
    fixed = rng.randrange(5)
    n = rng.choice([13, 14]) - 3*fixed

    c = [0]*34
    for t in rng.sample(range(136), n):
        c[t//4] += 1

    c = tuple(c)

    assert old_shanten(c, fixed) == faster_shanten(c, fixed), (
        "シャンテン不一致", c, fixed
    )

print("シャンテン数300件一致", flush=True)

for i in range(100):
    hand = rng.sample(range(136), 13)

    c = [0]*34
    for t in hand:
        c[t//4] += 1

    remaining = sorted(set(range(136))-set(hand))
    visible = rng.sample(remaining, 40)

    known = c.copy()
    for t in visible:
        known[t//4] += 1

    args = (
        tuple(c), 0,
        tuple(known), (0,)*34
    )

    assert old_ukeire(*args) == faster_ukeire(*args), (
        "受け入れ不一致", args
    )

print("受け入れ枚数100件一致", flush=True)


# 4. キャッシュ初期化

for fn in [
    block, merge, fast_partitions,
    legacy_partitions,
    old_shanten, old_ukeire,
    faster_shanten, faster_ukeire,
    acceptance_types, standard_score,
    shapes.improvement
]:
    fn.cache_clear()


# このプロセスの関数参照だけを変更
shapes.shanten = faster_shanten
options.shanten = faster_shanten
calls.shanten = faster_shanten

shapes.ukeire = faster_ukeire
options.ukeire = faster_ukeire
calls.ukeire = faster_ukeire


# 5. 元の6対局との完全一致比較

receipt = Path(sys.argv[1])
original_dir = Path(sys.argv[2])
profile_dir = Path(sys.argv[3])

original = json.loads(
    receipt.read_text(encoding="utf8")
)
config = original["config"]
ctx = prepare(config, "A")

expected = dict(
    signature=ctx["signature"],
    mode="A",
    config=config,
    ids=ctx["targets"].game_id.tolist()
)

assert original == expected

baseline = json.loads(
    (profile_dir/"fast_results.json").read_text()
)

baseline_times = {
    x["game_id"]: x["new_seconds"]
    for x in baseline
}

baseline_order = {
    x["game_id"]: i
    for i, x in enumerate(baseline)
}

rows = list(ctx["targets"].itertuples())

rows.sort(
    key=lambda r: baseline_order[str(r.game_id)]
)

assert len(rows) == 6

results = []

print("\n=== 実牌譜比較 ===", flush=True)

with zipfile.ZipFile(ctx["archive"]) as z:
    for index, r in enumerate(rows, 1):

        xml = checked_xml(z, r)
        t = time.perf_counter()

        result = extract(
            xml, config["player"],
            r.seat, r.dan, r.game_id,
            collect_trace=False
        )

        seconds = time.perf_counter()-t

        path = (
            original_dir/"traces"/
            (digest(r.game_id)+".json")
        )

        packet = json.loads(
            path.read_text(encoding="utf8")
        )

        assert packet["identity"]["game_id"] == r.game_id
        assert packet["identity"]["xml_sha256"] == r.xml_sha256

        old = packet["result"]["features"]
        new = result["features"]

        diff = [
            k for k in set(old)|set(new)
            if old.get(k) != new.get(k)
        ]

        before = baseline_times[str(r.game_id)]

        print(
            f"{index}/6 {r.game_id} "
            f"N9={before:.3f}秒 "
            f"N10={seconds:.3f}秒 "
            f"不一致={len(diff)}",
            flush=True
        )

        if diff:
            for k in sorted(diff)[:10]:
                print(k, old.get(k), new.get(k))
            raise RuntimeError("特徴量不一致")

        results.append(dict(
            game_id=str(r.game_id),
            dan=int(r.dan),
            N9_seconds=before,
            N10_seconds=seconds,
            speedup=before/seconds,
            differences=len(diff)
        ))

print("\n=== キャッシュ統計 ===")

for fn in [
    block, merge, standard_score,
    faster_shanten, acceptance_types,
    faster_ukeire, shapes.improvement
]:
    print(fn.__name__, fn.cache_info())

(profile_dir/"fast2_results.json").write_text(
    json.dumps(results, ensure_ascii=False),
    encoding="utf8"
)

print("6対局の一致確認完了")

''', encoding="utf8")

# 別プロセスで実行
process = subprocess.Popen(
    [
        sys.executable, "-u",
        str(FAST2_SCRIPT),
        str(RECEIPT),
        str(EXTRACTION_DIR),
        str(PROFILE_DIR)
    ],
    env=ENV,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf8",
    errors="replace"
)

for line in process.stdout:
    print(line, end="", flush=True)

if process.wait() != 0:
    raise RuntimeError("N10の検証に失敗しました")

df2 = pd.DataFrame(json.loads(
    (PROFILE_DIR/"fast2_results.json").read_text()
))

display(df2.round(3))

print("N9平均:", df2.N9_seconds.mean(), "秒")
print("N10平均:", df2.N10_seconds.mean(), "秒")
print(
    "追加高速化倍率:",
    df2.N9_seconds.sum()/df2.N10_seconds.sum()
)
print("不一致合計:", df2.differences.sum())
print(
    "全件推定時間（1プロセス）:",
    df2.N10_seconds.mean()*11573/3600,
    "時間"
)
print(
    "全件推定時間（2プロセス理想値）:",
    df2.N10_seconds.mean()*11573/7200,
    "時間"
)


=== 合成手牌テスト ===
シャンテン数300件一致
受け入れ枚数100件一致
A: 対象6 / キャッシュ候補6 / 新規解析予定0対局
実装90 / 保留10。保留項目を0で代用しません。
ZIP: /content/drive/MyDrive/卒論/mjlog_pf4-20_n30.zip
出力親: /content/drive/MyDrive/卒論/research_v2_features_100

=== 実牌譜比較 ===
1/6 2012062304gm-0029-0000-dff1fa7e N9=6.536秒 N10=3.373秒 不一致=0
2/6 2014061300gm-00a9-0000-95909518 N9=2.662秒 N10=2.225秒 不一致=0
3/6 2021050905gm-00a9-0000-6aeb09b8 N9=7.638秒 N10=7.614秒 不一致=0
4/6 2024062620gm-00a9-0000-4eb98c09 N9=4.475秒 N10=2.560秒 不一致=0
5/6 2025100518gm-0029-0000-8c403a29 N9=1.732秒 N10=1.502秒 不一致=0
6/6 2026051522gm-00a9-0000-0925ded1 N9=3.230秒 N10=2.555秒 不一致=0

=== キャッシュ統計 ===
block CacheInfo(hits=3878530, misses=14730, maxsize=100000, currsize=14730)
merge CacheInfo(hits=2885542, misses=2675, maxsize=100000, currsize=2675)
standard_score CacheInfo(hits=962227, misses=512, maxsize=50000, currsize=512)
faster_shanten CacheInfo(hits=455935, misses=962739, maxsize=60000, currsize=60000)
acceptance_types CacheInfo(hits=20802, misses=36447, maxsize=60000, cu

,game_id,dan,N9_seconds,N10_seconds,speedup,differences
0,2012062304gm-0029-0000-dff1fa7e,14,6.536,3.373,1.938,0
1,2014061300gm-00a9-0000-95909518,18,2.662,2.225,1.197,0
2,2021050905gm-00a9-0000-6aeb09b8,16,7.638,7.614,1.003,0
3,2024062620gm-00a9-0000-4eb98c09,18,4.475,2.560,1.748,0
4,2025100518gm-0029-0000-8c403a29,16,1.732,1.502,1.153,0
5,2026051522gm-00a9-0000-0925ded1,20,3.230,2.555,1.265,0


N9平均: 4.3786175508331935 秒
N10平均: 3.30475435949999 秒
追加高速化倍率: 1.3249449352404148
不一致合計: 0
全件推定時間（1プロセス）: 10.623867278470385 時間
全件推定時間（2プロセス理想値）: 5.311933639235193 時間


In [12]:
# N11：受け入れ判定の牌種別共通計算
# 既存コードと保存済み研究データは変更しない

FAST3_SCRIPT = PROFILE_DIR / "fast3_experiment.py"

prefix = FAST2_SCRIPT.read_text(
    encoding="utf8"
).split("# 3. 合成手牌で旧新版を検証", 1)[0]

FAST3_SCRIPT.write_text(prefix + r'''

acceptance_types_n10 = acceptance_types


@lru_cache(maxsize=30000)
def block_additions(c, suited):
    """牌種ブロックに1枚追加した場合の分解結果。"""
    out = []
    a = list(c)

    for i in range(len(c)):
        if c[i] >= 4:
            continue

        a[i] += 1
        out.append((i, block(tuple(a), suited)))
        a[i] -= 1

    return tuple(out)


@lru_cache(maxsize=50000)
def standard_acceptance(c, suited, rest, fixed, threshold):
    """他の3ブロックを固定した面子手の受け入れ。"""
    return tuple(
        i
        for i, parts in block_additions(c, suited)
        if standard_score(
            merge(parts, rest), fixed
        ) < threshold
    )


@lru_cache(maxsize=60000)
def acceptance_types(c, fixed):
    standard, pairs, orphans = faster_shanten(c, fixed)
    s = min(standard, pairs, orphans)

    groups = (
        c[:9], c[9:18], c[18:27], c[27:]
    )

    parts = tuple(
        block(g, i != 3)
        for i, g in enumerate(groups)
    )

    accepted = set()

    for i, g in enumerate(groups):
        others = [
            parts[j] for j in range(4) if j != i
        ]

        rest = merge(
            merge(others[0], others[1]), others[2]
        )

        for t in standard_acceptance(
            g, i != 3, rest, fixed, s
        ):
            accepted.add(i*9+t)

    # 七対子・国士無双の計算も維持
    if fixed == 0:
        distinct = 34-c.count(0)

        if pairs == s:
            for t, n in enumerate(c):
                if n == 1 or (n == 0 and distinct < 7):
                    accepted.add(t)

        if orphans == s:
            has_pair = any(
                c[t] >= 2 for t in shapes.TERMINALS
            )

            for t in shapes.TERMINALS:
                if c[t] == 0 or (
                    c[t] == 1 and not has_pair
                ):
                    accepted.add(t)

    return tuple(sorted(
        t for t in accepted if c[t] < 4
    ))


# ==========================================
# 1. N10とN11の構造的受け入れを直接比較
# ==========================================

print("=== 合成手牌テスト ===", flush=True)

rng = random.Random(20261126)

for i in range(1000):
    fixed = rng.randrange(5)
    n = 13 - 3*fixed

    c = [0]*34

    # 通常手牌に加え、染め手に近い偏った形も生成
    if i % 3 == 0:
        suit = rng.randrange(3)
        pool = list(range(suit*36, suit*36+36))
        pool += list(range(108, 136))
    else:
        pool = list(range(136))

    for t in rng.sample(pool, n):
        c[t//4] += 1

    c = tuple(c)

    old = acceptance_types_n10(c, fixed)
    new = acceptance_types(c, fixed)

    if old != new:
        raise RuntimeError(
            f"受け入れ牌種不一致: {c}, "
            f"fixed={fixed}, N10={old}, N11={new}"
        )

print("構造的受け入れ1000件すべて一致", flush=True)

# キャッシュを初期化
for fn in [
    block, merge, fast_partitions,
    legacy_partitions, old_shanten, old_ukeire,
    faster_shanten, faster_ukeire,
    acceptance_types, acceptance_types_n10,
    standard_score, block_additions,
    standard_acceptance, shapes.improvement
]:
    fn.cache_clear()

# 別プロセス内だけで高速版に差し替え
shapes.shanten = faster_shanten
options.shanten = faster_shanten
calls.shanten = faster_shanten

shapes.ukeire = faster_ukeire
options.ukeire = faster_ukeire
calls.ukeire = faster_ukeire


# ==========================================
# 2. 実牌譜6対局の比較
# ==========================================

receipt = Path(sys.argv[1])
original_dir = Path(sys.argv[2])
profile_dir = Path(sys.argv[3])

original = json.loads(
    receipt.read_text(encoding="utf8")
)

config = original["config"]
ctx = prepare(config, "A")

assert original == dict(
    signature=ctx["signature"],
    mode="A",
    config=config,
    ids=ctx["targets"].game_id.tolist()
)

baseline = json.loads(
    (profile_dir/"fast2_results.json").read_text()
)

times = {
    x["game_id"]: x["N10_seconds"] for x in baseline
}

order = {
    x["game_id"]: i for i, x in enumerate(baseline)
}

rows = list(ctx["targets"].itertuples())
rows.sort(key=lambda r: order[str(r.game_id)])

assert len(rows) == 6

results = []

print("\n=== 実牌譜6対局の比較 ===", flush=True)

with zipfile.ZipFile(ctx["archive"]) as z:
    for i, r in enumerate(rows, 1):

        xml = checked_xml(z, r)

        started = time.perf_counter()

        result = extract(
            xml, config["player"],
            r.seat, r.dan, r.game_id,
            collect_trace=False
        )

        elapsed = time.perf_counter()-started

        path = (
            original_dir/"traces"/
            (digest(r.game_id)+".json")
        )

        packet = json.loads(
            path.read_text(encoding="utf8")
        )

        assert packet["identity"]["game_id"] == r.game_id
        assert packet["identity"]["xml_sha256"] == r.xml_sha256

        old = packet["result"]["features"]
        new = result["features"]

        diff = [
            k for k in set(old)|set(new)
            if old.get(k) != new.get(k)
        ]

        before = times[str(r.game_id)]

        print(
            f"{i}/6 {r.game_id} "
            f"N10={before:.3f}秒 "
            f"N11={elapsed:.3f}秒 "
            f"不一致={len(diff)}",
            flush=True
        )

        if diff:
            for k in sorted(diff)[:10]:
                print(k, old.get(k), new.get(k))
            raise RuntimeError("特徴量不一致")

        results.append(dict(
            game_id=str(r.game_id),
            dan=int(r.dan),
            N10_seconds=before,
            N11_seconds=elapsed,
            speedup=before/elapsed,
            differences=len(diff)
        ))

print("\n=== キャッシュ統計 ===")

for fn in [
    block, merge, standard_score,
    block_additions, standard_acceptance,
    faster_shanten, acceptance_types,
    faster_ukeire, shapes.improvement
]:
    print(fn.__name__, fn.cache_info())

(profile_dir/"fast3_results.json").write_text(
    json.dumps(results, ensure_ascii=False),
    encoding="utf8"
)

print("6対局の一致確認完了")

''', encoding="utf8")


# 別プロセスで実行

process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        str(FAST3_SCRIPT),
        str(RECEIPT),
        str(EXTRACTION_DIR),
        str(PROFILE_DIR)
    ],
    env=ENV,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace"
)

for line in process.stdout:
    print(line, end="", flush=True)

if process.wait() != 0:
    raise RuntimeError("N11の検証に失敗しました")

df3 = pd.DataFrame(json.loads(
    (PROFILE_DIR/"fast3_results.json").read_text()
))

display(df3.round(3))

print("N10平均:", df3.N10_seconds.mean(), "秒")
print("N11平均:", df3.N11_seconds.mean(), "秒")
print(
    "追加高速化倍率:",
    df3.N10_seconds.sum()/df3.N11_seconds.sum()
)
print("不一致合計:", df3.differences.sum())

print(
    "全件推定時間（1プロセス）:",
    df3.N11_seconds.mean()*11573/3600, "時間"
)

print(
    "全件推定時間（2プロセス理想値）:",
    df3.N11_seconds.mean()*11573/7200, "時間"
)


=== 合成手牌テスト ===
構造的受け入れ1000件すべて一致
A: 対象6 / キャッシュ候補6 / 新規解析予定0対局
実装90 / 保留10。保留項目を0で代用しません。
ZIP: /content/drive/MyDrive/卒論/mjlog_pf4-20_n30.zip
出力親: /content/drive/MyDrive/卒論/research_v2_features_100

=== 実牌譜6対局の比較 ===
1/6 2012062304gm-0029-0000-dff1fa7e N10=3.373秒 N11=2.667秒 不一致=0
2/6 2014061300gm-00a9-0000-95909518 N10=2.225秒 N11=1.366秒 不一致=0
3/6 2021050905gm-00a9-0000-6aeb09b8 N10=7.614秒 N11=2.442秒 不一致=0
4/6 2024062620gm-00a9-0000-4eb98c09 N10=2.560秒 N11=1.526秒 不一致=0
5/6 2025100518gm-0029-0000-8c403a29 N10=1.502秒 N11=0.819秒 不一致=0
6/6 2026051522gm-00a9-0000-0925ded1 N10=2.555秒 N11=1.148秒 不一致=0

=== キャッシュ統計 ===
block CacheInfo(hits=537894, misses=14730, maxsize=100000, currsize=14730)
merge CacheInfo(hits=714696, misses=2988, maxsize=100000, currsize=2988)
standard_score CacheInfo(hits=258614, misses=512, maxsize=50000, currsize=512)
block_additions CacheInfo(hits=16801, misses=3467, maxsize=30000, currsize=3467)
standard_acceptance CacheInfo(hits=125520, misses=20268, maxsize=50000, c

,game_id,dan,N10_seconds,N11_seconds,speedup,differences
0,2012062304gm-0029-0000-dff1fa7e,14,3.373,2.667,1.265,0
1,2014061300gm-00a9-0000-95909518,18,2.225,1.366,1.629,0
2,2021050905gm-00a9-0000-6aeb09b8,16,7.614,2.442,3.118,0
3,2024062620gm-00a9-0000-4eb98c09,18,2.560,1.526,1.678,0
4,2025100518gm-0029-0000-8c403a29,16,1.502,0.819,1.834,0
5,2026051522gm-00a9-0000-0925ded1,20,2.555,1.148,2.226,0


N10平均: 3.30475435949999 秒
N11平均: 1.6611983043335385 秒
追加高速化倍率: 1.9893798054566612
不一致合計: 0
全件推定時間（1プロセス）: 5.340291104458901 時間
全件推定時間（2プロセス理想値）: 2.6701455522294504 時間


In [13]:
import shutil
from datetime import datetime
from uuid import uuid4

BACKUP_DIR = (
    ROOT / "research_v2_optimization_experiments" /
    (datetime.now().strftime("%Y%m%d_%H%M%S")
     + "_" + uuid4().hex[:8])
)

shutil.copytree(PROFILE_DIR, BACKUP_DIR)

print("高速化実験のバックアップ先:")
print(BACKUP_DIR)


高速化実験のバックアップ先:
/content/drive/MyDrive/卒論/research_v2_optimization_experiments/20261126_060717_c2674147


In [14]:
# N12：追加30対局の一致検証・並列処理
# 元の研究コード・保存済み特徴量は変更しない

import shutil
from datetime import datetime
from uuid import uuid4

# N11から高速化関数だけを取り出す
KERNEL = PROFILE_DIR / "fast_kernel.py"

kernel_source = FAST3_SCRIPT.read_text(
    encoding="utf8"
).split(
    "# 1. N10とN11の構造的受け入れを直接比較", 1
)[0]

kernel_source = kernel_source.replace(
    "shapes.partitions = fast_partitions\n", ""
)

KERNEL.write_text(kernel_source + '''

def install():
    shapes.partitions = fast_partitions

    shapes.shanten = faster_shanten
    options.shanten = faster_shanten
    calls.shanten = faster_shanten

    shapes.ukeire = faster_ukeire
    options.ukeire = faster_ukeire
    calls.ukeire = faster_ukeire

''', encoding="utf8")


BENCH_SCRIPT = PROFILE_DIR / "benchmark_30.py"

BENCH_SCRIPT.write_text(r'''
import sys
import json
import time
import zipfile
import unittest
import multiprocessing as mp
import numpy as np
import pandas as pd

from pathlib import Path
from types import SimpleNamespace
from concurrent.futures import ProcessPoolExecutor

from features_100.pipeline import prepare, checked_xml
from features_100.extract import extract
from decision_features.pipeline import select_pilot


def init_worker(archive, player, fast):
    global ZIP, PLAYER

    if fast:
        from fast_kernel import install
        install()

    ZIP = zipfile.ZipFile(archive)
    PLAYER = player


def worker(row):
    r = SimpleNamespace(**row)

    started = time.perf_counter()

    xml = checked_xml(ZIP, r)

    result = extract(
        xml, PLAYER,
        r.seat, r.dan, r.game_id,
        collect_trace=False
    )

    seconds = time.perf_counter() - started

    return dict(
        game_id=r.game_id,
        seconds=seconds,
        features=result["features"],
        audit=result["audit"],
        unknown_reasons=result["unknown_reasons"],
        round_counters=sorted(
            result["round_counters"],
            key=lambda x: (x["round"], x["feature_id"])
        )
    )


def main():
    mode = sys.argv[1]
    receipt = Path(sys.argv[2])
    output = Path(sys.argv[3])
    source = Path(sys.argv[4])

    if mode == "tests":
        from fast_kernel import install
        install()

        suite = unittest.defaultTestLoader.discover(
            str(source/"features_100/tests"),
            top_level_dir=str(source)
        )

        result = unittest.TextTestRunner(
            verbosity=1
        ).run(suite)

        if not result.wasSuccessful():
            raise RuntimeError("既存テスト失敗")

        print("高速版・既存テスト成功")
        return

    original = json.loads(
        receipt.read_text(encoding="utf8")
    )

    config = original["config"]
    ctx = prepare(config, "A")

    expected = dict(
        signature=ctx["signature"],
        mode="A",
        config=config,
        ids=ctx["targets"].game_id.tolist()
    )

    assert original == expected

    # これまでの6対局を除外
    d = ctx["d"]

    eligible = (
        d.hour.gt(pd.Timestamp(ctx["cutoff"]))
        & ~d.game_id.isin(ctx["shared"])
        & d.target_label.notna()
        & ~d.game_id.isin(original["ids"])
    )

    ids = []

    # 特徴量の値を見ず、時期を分散させて抽出
    for label in [0, 1]:
        group = d[
            eligible & d.target_label.eq(label)
        ].sort_values(["hour", "game_id"])

        assert len(group) >= 15

        positions = np.linspace(
            0, len(group)-1, 15, dtype=int
        )

        ids.extend(
            group.iloc[positions].game_id.tolist()
        )

    targets = select_pilot(
        d,
        pd.Timestamp(ctx["cutoff"]),
        ctx["shared"],
        per_group=3,
        explicit_ids=ids
    )

    assert len(targets) == 30

    rows = [
        dict(
            game_id=str(r.game_id),
            file_id=str(r.file_id),
            xml_sha256=str(r.xml_sha256),
            seat=int(r.seat),
            dan=int(r.dan)
        )
        for r in targets.itertuples()
    ]

    fast = mode != "old2"
    workers = 1 if mode == "fast1" else 2

    print(
        f"\n=== {mode}: "
        f"{len(rows)}対局 / "
        f"{workers}プロセス ===",
        flush=True
    )

    results = {}
    started = time.perf_counter()

    with ProcessPoolExecutor(
        max_workers=workers,
        mp_context=mp.get_context("fork"),
        initializer=init_worker,
        initargs=(
            str(ctx["archive"]),
            config["player"],
            fast
        )
    ) as pool:

        for i, result in enumerate(
            pool.map(worker, rows, chunksize=1), 1
        ):
            results[result["game_id"]] = result

            print(
                f'{i}/30 {result["game_id"]} '
                f'{result["seconds"]:.3f}秒',
                flush=True
            )

    elapsed = time.perf_counter() - started

    payload = dict(
        mode=mode,
        workers=workers,
        total_seconds=elapsed,
        results=results
    )

    (output/f"{mode}.json").write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            allow_nan=False
        ),
        encoding="utf8"
    )

    print(
        "全体処理時間:",
        round(elapsed, 3), "秒",
        flush=True
    )


if __name__ == "__main__":
    main()

''', encoding="utf8")


# 既存テスト → 旧版 → 高速版1・2プロセス
for mode in ["tests", "old2", "fast1", "fast2"]:

    print("\n" + "="*60)
    print("開始:", mode, flush=True)

    process = subprocess.Popen(
        [
            sys.executable,
            "-u",
            str(BENCH_SCRIPT),
            mode,
            str(RECEIPT),
            str(PROFILE_DIR),
            str(SOURCE)
        ],
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    if process.wait() != 0:
        raise RuntimeError(f"{mode}で失敗")


# ==========================================
# 旧版・新版の完全一致比較
# ==========================================

reports = {
    mode: json.loads(
        (PROFILE_DIR/f"{mode}.json").read_text()
    )
    for mode in ["old2", "fast1", "fast2"]
}

old = reports["old2"]["results"]

summary = []

for mode in ["old2", "fast1", "fast2"]:
    report = reports[mode]
    new = report["results"]

    assert set(old) == set(new)

    differences = []
    feature_differences = 0

    for game_id in old:
        a = old[game_id]
        b = new[game_id]

        for field in [
            "features",
            "audit",
            "unknown_reasons",
            "round_counters"
        ]:
            if a[field] != b[field]:
                differences.append((game_id, field))

        feature_differences += sum(
            a["features"].get(k) != b["features"].get(k)
            for k in set(a["features"]) | set(b["features"])
        )

    seconds = report["total_seconds"]

    summary.append(dict(
        mode=mode,
        workers=report["workers"],
        games=len(new),
        total_seconds=seconds,
        wall_seconds_per_game=seconds/len(new),
        feature_differences=feature_differences,
        audit_differences=len(differences),
        estimated_full_hours=(
            seconds/len(new)*11573/3600
        )
    ))

    if differences:
        print("不一致:", differences[:10])
        raise RuntimeError("一致検証失敗")


benchmark = pd.DataFrame(summary)

display(benchmark.round(3))

print(
    "\n旧版2プロセス → 新版2プロセス:",
    reports["old2"]["total_seconds"] /
    reports["fast2"]["total_seconds"],
    "倍"
)

print(
    "新版1 → 2プロセス:",
    reports["fast1"]["total_seconds"] /
    reports["fast2"]["total_seconds"],
    "倍"
)

print("全対局の一致検証完了")

benchmark.to_csv(
    PROFILE_DIR/"benchmark_summary.csv",
    index=False
)

# Driveにバックアップ
SAVE_DIR = (
    ROOT/"research_v2_optimization_experiments"/
    (
        "N12_" +
        datetime.now().strftime("%Y%m%d_%H%M%S") +
        "_" + uuid4().hex[:8]
    )
)

shutil.copytree(PROFILE_DIR, SAVE_DIR)

print("バックアップ先:", SAVE_DIR)



開始: tests
..................................................................................................................................................1/1 2012052419gm-test: ok 0.01s
保存先: /tmp/tmpzotg35bo/out/20261126T061249Z_854b6939
1/1 2012052419gm-test: ok 0.00s cache
保存先: /tmp/tmpzotg35bo/out/20261126T061249Z_cedf2a1a
1/1 2012052419gm-test: error 0.00s
保存先: /tmp/tmpzotg35bo/out/20261126T061249Z_636fdb88
..........................................................................................................
----------------------------------------------------------------------
Ran 252 tests in 2.162s

OK
高速版・既存テスト成功

開始: old2
A: 対象6 / キャッシュ候補6 / 新規解析予定0対局
実装90 / 保留10。保留項目を0で代用しません。
ZIP: /content/drive/MyDrive/卒論/mjlog_pf4-20_n30.zip
出力親: /content/drive/MyDrive/卒論/research_v2_features_100

=== old2: 30対局 / 2プロセス ===
1/30 2012062311gm-0029-0000-f8bbd45f 27.328秒
2/30 2012083011gm-00a9-0000-ddb8d450 15.349秒
3/30 2013032513gm-00a9-0000-c7356fb8 41.431秒
4/30 2013062509gm-00a9-000

,mode,workers,games,total_seconds,wall_seconds_per_game,feature_differences,audit_differences,estimated_full_hours
0,old2,2,30,425.835,14.195,0,0,45.631
1,fast1,1,30,55.192,1.840,0,0,5.914
2,fast2,2,30,46.268,1.542,0,0,4.958



旧版2プロセス → 新版2プロセス: 9.203586368217337 倍
新版1 → 2プロセス: 1.1928673175378055 倍
全対局の一致検証完了
バックアップ先: /content/drive/MyDrive/卒論/research_v2_optimization_experiments/N12_20261126_062147_a25d6f4d


In [15]:
# N13：高速版の正式な抽出処理への統合
# 旧版・既存研究結果は変更しない

import ast
import shutil
import hashlib
import tempfile
import json
import sys
import os
import subprocess
from pathlib import Path
from datetime import datetime
from uuid import uuid4

FAST_SOURCE = (
    Path(tempfile.mkdtemp(prefix="mahjong_fast_"))
    / "research_v2"
)

# 元の固定版を別の作業領域にコピー
for relative in FILES:
    target = FAST_SOURCE / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(SOURCE / relative, target)


# =====================================
# 1. N11の高速化関数を組み込む
# =====================================

kernel_text = KERNEL.read_text(encoding="utf8")
tree = ast.parse(kernel_text)
lines = kernel_text.splitlines()

definitions = {
    node.name: node
    for node in tree.body
    if isinstance(node, ast.FunctionDef)
}

names = [
    "prune",
    "block",
    "merge",
    "fast_partitions",
    "standard_score",
    "faster_shanten",
    "faster_ukeire",
    "block_additions",
    "standard_acceptance",
    "acceptance_types"
]

parts = []

for name in names:
    node = definitions[name]

    start = min(
        [node.lineno] +
        [x.lineno for x in node.decorator_list]
    )

    parts.append(
        "\n".join(lines[start-1:node.end_lineno])
    )

optimized_code = "\n\n".join(parts)
optimized_code = optimized_code.replace(
    "shapes.TERMINALS", "TERMINALS"
)

shape_path = FAST_SOURCE / "features_100/shapes.py"

shape_path.write_text(
    shape_path.read_text(encoding="utf8")
    + "\n\n# N9-N11 validated optimization\n"
    + optimized_code
    + """

partitions = fast_partitions
shanten = faster_shanten
ukeire = faster_ukeire
""",
    encoding="utf8"
)


# =====================================
# 2. 並列処理と保存処理の改善
# =====================================

pipeline_path = FAST_SOURCE / "features_100/pipeline.py"

src = pipeline_path.read_text(encoding="utf8")


def replace_once(text, old, new):
    if text.count(old) != 1:
        raise RuntimeError(
            "置換対象が想定と異なります: " + old[:100]
        )
    return text.replace(old, new, 1)


helpers = '''
from concurrent.futures import ProcessPoolExecutor
from collections import deque
from contextlib import nullcontext
from types import SimpleNamespace
import multiprocessing as mp


def _cache_path(cache, game_id, xml_hash, config):
    name = digest([game_id, xml_hash])
    if config.get("cache_shards", False):
        return cache / name[:2] / (name + ".json")
    return cache / (name + ".json")


def _init_worker(archive, player):
    global _worker_zip, _worker_player
    _worker_zip = zipfile.ZipFile(archive)
    _worker_player = player


def _extract_worker(row, collect_trace):
    r = SimpleNamespace(**row)
    started = time.perf_counter()

    result = extract(
        checked_xml(_worker_zip, r),
        _worker_player,
        r.seat,
        r.dan,
        r.game_id,
        collect_trace
    )

    return result, time.perf_counter() - started


def _parallel_jobs(ctx, pool):
    rows = enumerate(ctx["targets"].itertuples(), 1)

    if pool is None:
        for n, r in rows:
            yield n, r, None
        return

    pending = deque()

    def submit(n, r):
        path = _cache_path(
            ctx["cache"],
            r.game_id,
            r.xml_sha256,
            ctx["config"]
        )

        future = None

        if not path.exists():
            row = dict(
                game_id=r.game_id,
                file_id=r.file_id,
                xml_sha256=r.xml_sha256,
                seat=int(r.seat),
                dan=int(r.dan)
            )

            future = pool.submit(
                _extract_worker,
                row,
                ctx["mode"] == "A"
            )

        pending.append((n, r, future))

    for _ in range(
        2 * int(ctx["config"].get("workers", 1))
    ):
        item = next(rows, None)
        if item is not None:
            submit(*item)

    while pending:
        yield pending.popleft()

        item = next(rows, None)
        if item is not None:
            submit(*item)

'''

src = replace_once(
    src,
    "\ndef extract_run(ctx):",
    "\n" + helpers + "\ndef extract_run(ctx):"
)

src = replace_once(
    src,
    "paths=[cache/(digest([r.game_id,r.xml_sha256])+'.json') for r in targets.itertuples()]",
    "paths=[_cache_path(cache,r.game_id,r.xml_sha256,config) for r in targets.itertuples()]"
)

src = replace_once(
    src,
    """    with zipfile.ZipFile(ctx['archive']) as z:
        for n,r in enumerate(ctx['targets'].itertuples(),1):""",
    """    workers=int(ctx['config'].get('workers',1))
    if workers not in (1,2,4):
        raise ValueError('workers must be 1, 2 or 4')

    executor = (
        ProcessPoolExecutor(
            max_workers=workers,
            mp_context=mp.get_context('fork'),
            initializer=_init_worker,
            initargs=(
                str(ctx['archive']),
                ctx['config']['player']
            )
        )
        if workers>1 else nullcontext(None)
    )

    with zipfile.ZipFile(ctx['archive']) as z, executor as pool:
        for n,r,future in _parallel_jobs(ctx,pool):"""
)

src = replace_once(
    src,
    "started=time.perf_counter(); reused=False; trace_only=False",
    "started=time.perf_counter(); reused=False; trace_only=False; worker_seconds=None"
)

src = replace_once(
    src,
    "path=ctx['cache']/(digest([r.game_id,r.xml_sha256])+'.json')",
    "path=_cache_path(ctx['cache'],r.game_id,r.xml_sha256,ctx['config'])"
)

src = replace_once(
    src,
    """                    result=extract(checked_xml(z,r),identity['player'],r.seat,r.dan,r.game_id,ctx['mode']=='A')""",
    """                    if future is None:
                        result=extract(checked_xml(z,r),identity['player'],r.seat,r.dan,r.game_id,ctx['mode']=='A')
                    else:
                        result,worker_seconds=future.result()"""
)

src = replace_once(
    src,
    "lock=path.with_suffix('.lock'); lock.mkdir(exist_ok=False)",
    "path.parent.mkdir(parents=True,exist_ok=True)\n                    lock=path.with_suffix('.lock'); lock.mkdir(exist_ok=False)"
)

src = replace_once(
    src,
    "seconds=time.perf_counter()-started,rule_profile_verified=",
    "seconds=time.perf_counter()-started,worker_seconds=worker_seconds,rule_profile_verified="
)

src = replace_once(
    src,
    """            write_csv(out/'extraction_audit.csv',pd.DataFrame(audit))""",
    """            if n%50==0 or n==len(ctx['targets']):
                write_csv(out/'extraction_audit.csv',pd.DataFrame(audit))"""
)

pipeline_path.write_text(src, encoding="utf8")

# Python構文チェック
for path in FAST_SOURCE.rglob("*.py"):
    ast.parse(path.read_text(encoding="utf8"))

FAST_ENV = dict(
    ENV,
    PYTHONPATH=os.pathsep.join(map(str, [
        DEPS,
        FAST_SOURCE,
        FAST_SOURCE/"choices",
        FAST_SOURCE/"smoothing"
    ]))
)


def fast_command(args):
    process = subprocess.Popen(
        [sys.executable, "-u", *map(str, args)],
        env=FAST_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf8",
        errors="replace"
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    if process.wait() != 0:
        raise RuntimeError("高速版の実行失敗")


# =====================================
# 3. 統合後の既存テスト
# =====================================

fast_command([
    "-m", "unittest", "discover",
    "-s", FAST_SOURCE/"features_100/tests",
    "-t", FAST_SOURCE,
    "-q"
])


# =====================================
# 4. 高速版ソースをDriveへ保存
# =====================================

FAST_VERSION_DIR = (
    ROOT/"research_v2_fast_versions"/
    (
        datetime.now().strftime("%Y%m%d_%H%M%S")
        + "_" + uuid4().hex[:8]
    )
)

shutil.copytree(
    FAST_SOURCE,
    FAST_VERSION_DIR/"research_v2",
    ignore=shutil.ignore_patterns("__pycache__")
)

FAST_OUTPUT_ROOT = ROOT/"research_v2_features_100_fast"

original_config = json.loads(
    RECEIPT.read_text(encoding="utf8")
)["config"]

fast_config = dict(original_config)

fast_config.update(
    output_root=str(FAST_OUTPUT_ROOT),
    workers=2,
    cache_shards=True
)

(FAST_VERSION_DIR/"config_base.json").write_text(
    json.dumps(fast_config, ensure_ascii=False, indent=2),
    encoding="utf8"
)

print("\n高速版保存先:", FAST_VERSION_DIR)


# =====================================
# 5. N12と同じ30対局で保存テスト
# =====================================

reference = json.loads(
    (PROFILE_DIR/"old2.json").read_text(
        encoding="utf8"
    )
)["results"]

smoke_config = dict(fast_config)
smoke_config["pilot_ids"] = list(reference)

SMOKE_CONFIG = FAST_SOURCE/"smoke_config.json"
SMOKE_RECEIPT = FAST_SOURCE/"smoke_receipt.json"

SMOKE_CONFIG.write_text(
    json.dumps(smoke_config, ensure_ascii=False),
    encoding="utf8"
)


def smoke_run():
    for action in ["preview", "extract"]:
        fast_command([
            "-m", "features_100.cli",
            action, SMOKE_CONFIG,
            "--mode", "A",
            "--receipt", SMOKE_RECEIPT
        ])

    return Path(json.loads(
        SMOKE_RECEIPT.with_suffix(
            ".result.json"
        ).read_text(encoding="utf8")
    )["output"])


SMOKE_DIR = smoke_run()


# =====================================
# 6. 保存結果と旧版の一致確認
# =====================================

import pandas as pd


def json_digest(value):
    return hashlib.sha256(
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True
        ).encode()
    ).hexdigest()


def verify_saved(directory):
    audit = pd.read_csv(
        directory/"extraction_audit.csv"
    )

    sources = json.loads(
        (directory/"sources.json").read_text(
            encoding="utf8"
        )
    )

    marker = json.loads(
        (directory/"complete.json").read_text(
            encoding="utf8"
        )
    )

    assert marker["all_reconstructed"]
    assert len(audit) == 30
    assert audit.game_id.is_unique
    assert set(audit.game_id) == set(reference)

    differences = []

    for r in audit.itertuples():
        key = json_digest([
            r.game_id, r.xml_sha256
        ])

        path = (
            Path(sources["cache"])
            /key[:2]/(key+".json")
        )

        packet = json.loads(
            path.read_text(encoding="utf8")
        )

        assert packet["signature"] == marker["signature"]
        assert packet["checksum"] == json_digest(
            packet["payload"]
        )

        assert packet["payload"]["identity"]["game_id"] == r.game_id

        result = packet["payload"]["result"]
        old = reference[r.game_id]

        for field in [
            "features", "audit", "unknown_reasons"
        ]:
            if result[field] != old[field]:
                differences.append((r.game_id, field))

        rounds = sorted(
            result["round_counters"],
            key=lambda x: (x["round"], x["feature_id"])
        )

        if rounds != old["round_counters"]:
            differences.append(
                (r.game_id, "round_counters")
            )

    print("保存結果の不一致:", len(differences))

    if differences:
        print(differences[:10])
        raise RuntimeError("旧版との不一致")

    return audit


first_audit = verify_saved(SMOKE_DIR)

print("\n初回保存テスト成功")


# =====================================
# 7. 再実行時のキャッシュ再利用検証
# =====================================

print("\n=== 再開・キャッシュテスト ===")

RESUME_DIR = smoke_run()
resume_audit = verify_saved(RESUME_DIR)

assert resume_audit["reused"].eq(True).all()

print(
    "キャッシュ再利用:",
    int(resume_audit["reused"].sum()),
    "/ 30対局"
)

print("\nN13の全検証に成功しました")
print("高速版保存先:", FAST_VERSION_DIR)
print("高速版出力先:", FAST_OUTPUT_ROOT)
print("初回保存結果:", SMOKE_DIR)
print("再実行結果:", RESUME_DIR)


1/1 2012052419gm-test: ok 0.01s
保存先: /tmp/tmpckrbe9eo/out/20261126T062846Z_0cea39ff
1/1 2012052419gm-test: ok 0.00s cache
保存先: /tmp/tmpckrbe9eo/out/20261126T062846Z_fe5b5565
1/1 2012052419gm-test: error 0.00s
保存先: /tmp/tmpckrbe9eo/out/20261126T062847Z_f10fd009
----------------------------------------------------------------------
Ran 252 tests in 2.311s

OK

高速版保存先: /content/drive/MyDrive/卒論/research_v2_fast_versions/20261126_062849_43a1d6c3
A: 対象30 / キャッシュ候補0 / 新規解析予定30対局
実装90 / 保留10。保留項目を0で代用しません。
ZIP: /content/drive/MyDrive/卒論/mjlog_pf4-20_n30.zip
出力親: /content/drive/MyDrive/卒論/research_v2_features_100_fast
A: 対象30 / キャッシュ候補0 / 新規解析予定30対局
実装90 / 保留10。保留項目を0で代用しません。
ZIP: /content/drive/MyDrive/卒論/mjlog_pf4-20_n30.zip
出力親: /content/drive/MyDrive/卒論/research_v2_features_100_fast
1/30 2012062311gm-0029-0000-f8bbd45f: ok 4.89s
2/30 2012083011gm-00a9-0000-ddb8d450: ok 1.18s
3/30 2013032513gm-00a9-0000-c7356fb8: ok 1.77s
4/30 2013062509gm-00a9-0000-95b04e2f: ok 2.22s
5/30 2014061320gm-00a9

In [16]:
# N14-a：高速版による全件解析の準備

import sys
import os
import json
import shutil
import tempfile
import subprocess
import time
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/卒論")

FAST_VERSION_DIR = (
    ROOT / "research_v2_fast_versions"
    / "20261126_062849_43a1d6c3"
)

assert FAST_VERSION_DIR.exists()
assert DEPS.exists(), "先にN1を実行してください"

# 保存済み高速版をローカル領域に復元
FULL_WORK = Path(tempfile.mkdtemp(
    prefix="mahjong_full_"
))

FULL_SOURCE = FULL_WORK / "research_v2"

shutil.copytree(
    FAST_VERSION_DIR / "research_v2",
    FULL_SOURCE
)

FULL_ENV = dict(
    os.environ,
    PYTHONPATH=os.pathsep.join(map(str, [
        DEPS,
        FULL_SOURCE,
        FULL_SOURCE / "choices",
        FULL_SOURCE / "smoothing"
    ])),
    PYTHONIOENCODING="utf-8",
    MPLBACKEND="Agg"
)

# 高速版の設定を読み込む
full_config = json.loads(
    (FAST_VERSION_DIR / "config_base.json")
    .read_text(encoding="utf8")
)

full_config.update(
    workers=2,
    cache_shards=True,
    pilot_ids=None
)

FULL_CONFIG = FULL_WORK / "config.json"
FULL_RECEIPT = FULL_WORK / "preview_B.json"

FULL_CONFIG.write_text(
    json.dumps(
        full_config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf8"
)

# 全件解析の対象を確認
result = subprocess.run(
    [
        sys.executable,
        "-u",
        "-m", "features_100.cli",
        "preview",
        str(FULL_CONFIG),
        "--mode", "B",
        "--receipt", str(FULL_RECEIPT)
    ],
    env=FULL_ENV,
    capture_output=True,
    text=True,
    encoding="utf8"
)

print(result.stdout)
print(result.stderr)
result.check_returncode()

preview = json.loads(
    FULL_RECEIPT.read_text(encoding="utf8")
)

# N13で検証済みの版と一致するか確認
smoke_marker_path = (
    ROOT / "research_v2_features_100_fast"
    / "20261126T062855Z_4802a48a"
    / "complete.json"
)

smoke_marker = json.loads(
    smoke_marker_path.read_text(encoding="utf8")
)

assert preview["signature"] == smoke_marker["signature"], (
    "N13で検証した版と署名が一致しません"
)

TOTAL_GAMES = len(preview["ids"])

print("検証済み高速版との署名一致：OK")
print("全件解析対象:", TOTAL_GAMES, "対局")
print("使用CPUプロセス:", full_config["workers"])
print("出力先:", full_config["output_root"])
print("全件解析の準備完了")


B: 対象11573 / キャッシュ候補30 / 新規解析予定11543対局
実装90 / 保留10。保留項目を0で代用しません。
ZIP: /content/drive/MyDrive/卒論/mjlog_pf4-20_n30.zip
出力親: /content/drive/MyDrive/卒論/research_v2_features_100_fast


検証済み高速版との署名一致：OK
全件解析対象: 11573 対局
使用CPUプロセス: 2
出力先: /content/drive/MyDrive/卒論/research_v2_features_100_fast
全件解析の準備完了


In [17]:
# N14-b：全件解析
# 成功した対局はDriveへ順次保存される

import re
import pandas as pd

started = time.perf_counter()

print("全件解析開始")
print("対象:", TOTAL_GAMES, "対局")
print("中断後も成功キャッシュを再利用できます")

process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "-m", "features_100.cli",
        "extract",
        str(FULL_CONFIG),
        "--mode", "B",
        "--receipt", str(FULL_RECEIPT)
    ],
    env=FULL_ENV,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf8",
    errors="replace"
)

for line in process.stdout:
    match = re.match(r"^(\d+)/(\d+)\s", line)

    if match:
        done = int(match.group(1))
        total = int(match.group(2))

        # 最初の5対局と50対局ごとの進捗を表示
        if (
            done <= 5
            or done % 50 == 0
            or done == total
            or "error" in line.lower()
        ):
            print(line, end="", flush=True)

            if done % 50 == 0:
                elapsed = time.perf_counter() - started

                print(
                    f"進捗: {done/total:.2%} / "
                    f"経過: {elapsed/60:.1f}分",
                    flush=True
                )
    else:
        print(line, end="", flush=True)

returncode = process.wait()

if returncode != 0:
    raise RuntimeError(
        f"抽出処理が中断・失敗しました: {returncode}"
    )

FULL_EXTRACTION_DIR = Path(
    json.loads(
        FULL_RECEIPT.with_suffix(".result.json")
        .read_text(encoding="utf8")
    )["output"]
)

marker = json.loads(
    (FULL_EXTRACTION_DIR / "complete.json")
    .read_text(encoding="utf8")
)

audit = pd.read_csv(
    FULL_EXTRACTION_DIR / "extraction_audit.csv"
)

print("\n=== 全件解析結果 ===")
print("対象対局:", len(audit))
print("成功:", int(audit.status.eq("ok").sum()))
print("失敗:", int(audit.status.ne("ok").sum()))
print("キャッシュ再利用:", int(audit.reused.eq(True).sum()))
print("全件復元成功:", marker["all_reconstructed"])
print("処理時間:", (time.perf_counter()-started)/3600, "時間")
print("保存先:", FULL_EXTRACTION_DIR)

if not marker["all_reconstructed"]:
    print("\n失敗対局を確認してください")
    display(audit[audit.status.ne("ok")])
else:
    print("\n全件抽出成功")
    display(pd.read_csv(
        FULL_EXTRACTION_DIR / "feature_audit.csv"
    ))


全件解析開始
対象: 11573 対局
中断後も成功キャッシュを再利用できます
B: 対象11573 / キャッシュ候補30 / 新規解析予定11543対局
実装90 / 保留10。保留項目を0で代用しません。
ZIP: /content/drive/MyDrive/卒論/mjlog_pf4-20_n30.zip
出力親: /content/drive/MyDrive/卒論/research_v2_features_100_fast
1/11573 2011011315gm-0009-0000-2508363c: ok 6.67s
2/11573 2011011315gm-0009-0000-f3a2c85c: ok 0.07s
3/11573 2011011316gm-0009-0000-10848bb1: ok 0.95s
4/11573 2011011317gm-0009-0000-afab4951: ok 2.42s
5/11573 2011011322gm-0009-0000-a0ef1c38: ok 0.65s
50/11573 2011042523gm-0009-0000-8848e73d: ok 0.82s
進捗: 0.43% / 経過: 1.6分


KeyboardInterrupt: 

## N6：C・保存済み分子分母から品質評価（ZIPを読まない）

全件完了したBのフォルダを指定します。探索期間末を事前に決め、以降の窓は特徴量統計から除外します。
過去に確認済みのデータを「未使用」に戻すことはできません。10/20対局は時間帯を分割せず、同一段位群・非重複で作成。
原対局が共通の感度分析も出力します。比率は分子合計÷分母合計、窓を等重みで比較します。
親比較の対応状況はparent_coverage.csvを参照してください。全100候補の親比較は未完です。


In [ ]:
# セッション再開時は、Bで出力された実在フォルダを指定してください。
FULL_EXTRACTION_DIR = EXTRACTION_DIR
if MODE != 'B':
    print('AではCを実行しません。監査後にMODE=BでN2〜N5を実行してください。')
else:
    if not EXPLORATION_END_HOUR:
        raise ValueError('N2のEXPLORATION_END_HOURを設定してください。未使用評価期間を残す境界が必要です。')
    save_config()
    run_step('quality',['--extraction-dir',str(FULL_EXTRACTION_DIR)])
    QUALITY_DIR = Path(json.loads(RECEIPT.with_suffix('.result.json').read_text())['output'])
    print('品質評価の保存先:',QUALITY_DIR)
    display(pd.read_csv(QUALITY_DIR/'feature_comparison.csv'))
    display(Markdown((QUALITY_DIR/'report_ja.md').read_text(encoding='utf8')))
